# 01 — Data Understanding

**Day 1, Step 1.** Load every file and look at it. No modelling today.

The same checklist runs over all five datasets so the results are comparable
rather than ad hoc: shape, dtypes, missingness, duplicates, and — most
importantly — candidate join keys.

> *Note to self from the Build Notes:* don't merge anything yet, even if two
> files look like they share an ID. A matching column name isn't proof of a
> matching key.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


## The five source datasets

In [2]:
from hrai.profiling.profiler import profile_dataset

DATASETS = ["employee_attrition", "hr_performance_engagement",
            "occupation_data", "essential_skills", "software_skills"]

frames, profiles = {}, {}
for name in DATASETS:
    frames[name] = load_raw(name)
    profiles[name] = profile_dataset(frames[name], name)

pd.DataFrame([{
    "dataset": p.name, "rows": p.rows, "cols": p.columns,
    "candidate_key": ", ".join(p.candidate_keys) or "none",
    "missing_cells": p.total_missing_cells, "duplicate_rows": p.duplicate_rows,
    "constant_cols": ", ".join(p.constant_columns) or "none",
} for p in profiles.values()])

2026-08-28 01:54:10 | INFO  | raw dataset loaded


2026-08-28 01:54:10 | INFO  | dataset profiled


2026-08-28 01:54:10 | INFO  | raw dataset loaded


2026-08-28 01:54:10 | INFO  | dataset profiled


2026-08-28 01:54:10 | INFO  | raw dataset loaded


...

2026-08-28 01:54:10 | INFO  | raw dataset loaded


2026-08-28 01:54:10 | INFO  | dataset profiled


2026-08-28 01:54:10 | INFO  | raw dataset loaded


2026-08-28 01:54:10 | INFO  | dataset profiled


,dataset,rows,cols,candidate_key,missing_cells,duplicate_rows,constant_cols
0,employee_attrition,1470,35,EmployeeNumber,0,0,"EmployeeCount, Over18, StandardHours"
1,hr_performance_engagement,3150,39,none,3088,150,none
2,occupation_data,1016,3,"O*NET-SOC Code, Title, Description",0,0,none
3,essential_skills,18200,15,none,9100,0,"N, Domain Source"
4,software_skills,31821,7,none,0,0,none


## The target, and why accuracy is the wrong metric

237 of 1,470 employees left — 16.1%. A model that predicts "stays" for everyone
scores 83.9% accuracy while being completely useless, which is why every metric
downstream is precision / recall / F1 / ROC-AUC / PR-AUC and never accuracy.

In [3]:
att = frames["employee_attrition"]
print(att["Attrition"].value_counts())
print()
print((att["Attrition"].value_counts(normalize=True) * 100).round(2))

Attrition
No     1233
Yes     237
Name: count, dtype: int64

Attrition
No     83.88
Yes    16.12
Name: proportion, dtype: float64


## Hunting for a join key

The Build Notes' heuristic: any column with 'id' in the name is a candidate.

In [4]:
for name, frame in frames.items():
    print(f"{name:30} {[c for c in frame.columns if 'id' in c.lower()]}")

employee_attrition             []
hr_performance_engagement      ['Employee ID']
occupation_data                []
essential_skills               ['Element ID', 'Scale ID']
software_skills                ['Element ID']


## The critical test — do the shared IDs mean the same people?

`employee_attrition.EmployeeNumber` and `hr_performance_engagement."Employee ID"`
overlap on 753 values. That looks joinable. Before believing it, compare
attributes that should agree if the rows describe the same person.

In [5]:
from hrai.profiling.profiler import key_overlap_evidence

eng = frames["hr_performance_engagement"].copy()
dob = pd.to_datetime(eng["DOB"], format="%d-%m-%Y", errors="coerce")
age = 2023 - dob.dt.year
eng["_age_from_dob"] = age.where(age > 0, age + 100)

evidence = key_overlap_evidence(
    frames["employee_attrition"], eng, "EmployeeNumber", "Employee ID",
    {"Gender": "GenderCode", "Age": "_age_from_dob"}, tolerance={"Age": 1.0},
)
for label, detail in evidence["attribute_agreement"].items():
    print(f"{label:26} {detail['agree']:>5}/{detail['compared']:<5} = {detail['agree_pct']:>5}%")
print()
print("VERDICT:", evidence["verdict"])

Gender vs GenderCode         366/753   =  48.6%
Age vs _age_from_dob          45/753   =   6.0%

VERDICT: COINCIDENTAL OVERLAP — attributes agree only 27.3% on the 753 shared key values. These are different populations; joining them would fabricate records.


### Finding F1 — these are two different companies

Gender agrees 48.6% of the time. That is a coin flip. Age agrees 6% of the time.

If these were the same employees, both would be near 100%. They are two
unrelated HR datasets whose ID ranges happen to overlap. **Joining them would
fabricate 753 people who do not exist.**

This drives the whole architecture — see `docs/adr/001-two-population-architecture.md`.

## What each dataset is for

| Dataset | Purpose | Finding |
|---|---|---|
| `employee_attrition` | Feeds the attrition model directly | Clean: 0 missing, 0 duplicates |
| `hr_performance_engagement` | Engagement analytics | **Event grain**, not employee grain (F5) |
| `occupation_data` | Role master reference table | No exact title match to our roles (F4) |
| `essential_skills` | **Tier 1** foundational skills | Only 10 cognitive skills, graded IM/LV (F2) |
| `software_skills` | **Tier 2** technical tools | 8,753 tools, Hot Technology flags (F2) |

In [6]:
ess = frames["essential_skills"]
print("Foundational skills (all of them):")
print(sorted(ess["Element Name"].unique()))
print()
print("Scales:", dict(ess["Scale ID"].value_counts()))
sw = frames["software_skills"]
print(f"\nTechnical: {sw['Workplace Example'].nunique():,} tools "
      f"across {sw['Element Name'].nunique()} categories")

Foundational skills (all of them):
['Active Learning', 'Active Listening', 'Critical Thinking', 'Learning Strategies', 'Mathematics', 'Monitoring', 'Reading Comprehension', 'Science', 'Speaking', 'Writing']

Scales: {'IM': np.int64(9100), 'LV': np.int64(9100)}

Technical: 8,753 tools across 134 categories


Run `make profile` to regenerate `docs/data_dictionary.md` and
`docs/findings.json` from these same functions.